In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/dual-aging-diffusion").resolve()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", Path.cwd())
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)
print("data exists:", (PROJECT_ROOT / "data").exists())
print("create_data exists:", (PROJECT_ROOT / "data" / "create_data.py").exists())

cwd: /workspace/dual-aging-diffusion
PROJECT_ROOT in sys.path: True
data exists: True
create_data exists: True


In [2]:
from pathlib import Path
import torch

from data.create_data import (
    build_local_dataloaders,
    build_global_dataloaders,
    build_local_fused_dataloaders,
    build_single_person_sampling_loader,
)


import data.global_path_datasets as global_paths


from src.diffusion_pipeline.load_diffusion_models import (
    build_global_local_bundles,
    build_mixed_lora_dora_training_setup,
)
from src.score_net.load_scorenet import load_score_net_safely
from src.loss.local_loss import LDLALocalAgingLoss
from src.loss.global_loss import GlobalAgingLoss
from src.loss.global_aux_bundle import GlobalLossAuxBundle
from src.training.train_aging_model import train_global_local_face_aging
from src.training.mixed_precision import resolve_device, get_effective_amp_dtype

Device: cuda
Dtype: torch.float16
Device: cuda
Dtype: torch.float16


In [3]:
# Runtime
device = resolve_device("auto")
amp_enabled = True
amp_dtype = "bf16"
dtype = get_effective_amp_dtype(amp_dtype=amp_dtype, device=device) or torch.float32

run_name = "notebook_global_local_run"
checkpoint_root = "training_checkpoints/notebook_global_local_run"

In [7]:
# Global data paths
PROJECT_ROOT = Path("/workspace/dual-aging-diffusion")

# Carpeta donde el pipeline va a dejar/leer las imágenes extraídas
global_paths.GLOBAL_IMAGE_DIR = PROJECT_ROOT / "data" / "global_extracted"

# CSV del repo
global_paths.GLOBAL_CSV_PATH = PROJECT_ROOT / "data" / "ffhq_predictions" / "ffhq_face_attribute_prompts.csv"

# ZIPs reales en Vast
global_paths.DRIVE_ZIPS = [
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-001.zip",
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-002.zip",
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-003.zip",
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-004.zip",
]

print("GLOBAL_IMAGE_DIR:", global_paths.GLOBAL_IMAGE_DIR)
print("GLOBAL_CSV_PATH exists:", global_paths.GLOBAL_CSV_PATH.exists())

for z in global_paths.DRIVE_ZIPS:
    print(z.name, z.exists(), round(z.stat().st_size / 1024**3, 2) if z.exists() else None, "GB")

GLOBAL_IMAGE_DIR: /workspace/dual-aging-diffusion/data/global_extracted
GLOBAL_CSV_PATH exists: True
ffhq_subset_6k_extremes-001.zip True 2.0 GB
ffhq_subset_6k_extremes-002.zip True 2.0 GB
ffhq_subset_6k_extremes-003.zip True 2.0 GB
ffhq_subset_6k_extremes-004.zip True 1.07 GB


In [72]:
skip_zones = ["labio_superior"]


# Base local crop-level random loader.
# Used by the normal local LDLA loss.
local_objects = build_local_dataloaders(
    batch_size=4,
    num_workers=4,
    pin_memory=True, skip=skip_zones,
)

# Global full-face loader.
# Used by the global branch.
global_objects = build_global_dataloaders(
    batch_size=4,
    num_workers=4,
    pin_memory=True, skip=skip_zones,
)

# Aligned image-level fused-loss loader.
# Used only if use_fused_loss=True.
local_fused_objects = build_local_fused_dataloaders(
    batch_size=1,
    num_workers=4,
    pin_memory=True,
    max_crops_per_image=None,  # None = all available local crops per image
    skip=skip_zones,
)

# Fixed one-person monitoring loader.
# Same loader is passed as global and local sampling loader.
sampling_objects = build_single_person_sampling_loader(
    image_stem="09501",
    target_age=75,
    skip=skip_zones,
    local_target_score={
        "default": 85.0,
        "frente": 85.0,
        "surcos_nasogenianos": 85.0,
        "bajo_ojo_ojeras": 85.0,
        "patas_de_gallo": 85.0,
    },
    num_workers=2,
    pin_memory=False)

local_train_loader = local_objects["train_loader"]
global_train_loader = global_objects["train_loader"]
local_fused_train_loader = local_fused_objects["train_loader"]

monitor_loader = sampling_objects["loader"]
sampling_loader_global = monitor_loader
sampling_loader_local = monitor_loader

[OK] JSON annotations already extracted: 56
[OK] Images already extracted: 56
[OK] Indexed unique images: 56
[OK] JSON files found: 56
[OK] Local annotated crop samples: 591
[OK] Unique images: 56
[OK] Train images: 48 | train crops: 503
[OK] Val images:   8 | val crops:   88

========== LOCAL SCORE SAMPLER ==========
Dataset virtual length: 4640
Real samples:           464
Global high share >=75: 9.70%
Anatomical prior:       True

[Region high-score shares]
bajo_ojo_ojeras                  n=  92 >=75=  7 share=  7.61% scarcity_factor=1.075 anatomical_factor=1.250
comisuras_lineas_marioneta       n=  85 >=75= 11 share= 12.94% scarcity_factor=1.000 anatomical_factor=1.250
frente                           n=  44 >=75=  2 share=  4.55% scarcity_factor=1.186 anatomical_factor=1.350
glabela_entrecejo                n=  47 >=75=  6 share= 12.77% scarcity_factor=1.000 anatomical_factor=1.050
patas_de_gallo                   n=  59 >=75=  7 share= 11.86% scarcity_factor=1.000 anatomical_fact

In [73]:
# Load diffusion models
global_bundle, local_bundle = build_global_local_bundles(
    global_model_id="SG161222/Realistic_Vision_V6.0_B1_noVAE",
    global_vae_id="stabilityai/sd-vae-ft-mse",
    local_model_id="SG161222/Realistic_Vision_V6.0_B1_noVAE",
    local_vae_id=None,
    device=device,
    dtype=dtype,
    print_memory=True,
)


BUILDING GLOBAL + LOCAL BUNDLES
Global model id: SG161222/Realistic_Vision_V6.0_B1_noVAE
Global VAE id: stabilityai/sd-vae-ft-mse
Global uses external VAE: True
--------------------------------------------------------------------------------
Local model id: SG161222/Realistic_Vision_V6.0_B1_noVAE
Local VAE id: stabilityai/sd-vae-ft-mse
Local uses external VAE: True

[Loading diffusion components]
Model id: SG161222/Realistic_Vision_V6.0_B1_noVAE
Use external VAE: True


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
An error occurred while trying to fetch SG161222/Realistic_Vision_V6.0_B1_noVAE: SG161222/Realistic_Vision_V6.0_B1_noVAE does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading external VAE: stabilityai/sd-vae-ft-mse

[Loaded Components]
Tokenizer: <class 'transformers.models.clip.tokenization_clip.CLIPTokenizer'>
Text encoder: <class 'transformers.models.clip.modeling_clip.CLIPTextModel'>
UNet: <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>
Scheduler train: <class 'diffusers.schedulers.scheduling_ddpm.DDPMScheduler'>
Scheduler infer: <class 'diffusers.schedulers.scheduling_ddim.DDIMScheduler'>
VAE: <class 'diffusers.models.autoencoders.autoencoder_kl.AutoencoderKL'>

[Config Checks]
Bundle model id: SG161222/Realistic_Vision_V6.0_B1_noVAE
Bundle VAE id: stabilityai/sd-vae-ft-mse
VAE scaling factor: 0.18215
UNet in_channels: 4
UNet cross_attention_dim: 768
Scheduler num_train_timesteps: 1000
[After loading GLOBAL bundle] allocated=8.19 GB | reserved=8.25 GB | max=8.19 GB

[Loading diffusion components]
Model id: SG161222/Realistic_Vision_V6.0_B1_noVAE
Use external VAE: True


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch SG161222/Realistic_Vision_V6.0_B1_noVAE: SG161222/Realistic_Vision_V6.0_B1_noVAE does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading external VAE: stabilityai/sd-vae-ft-mse

[Loaded Components]
Tokenizer: <class 'transformers.models.clip.tokenization_clip.CLIPTokenizer'>
Text encoder: <class 'transformers.models.clip.modeling_clip.CLIPTextModel'>
UNet: <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>
Scheduler train: <class 'diffusers.schedulers.scheduling_ddpm.DDPMScheduler'>
Scheduler infer: <class 'diffusers.schedulers.scheduling_ddim.DDIMScheduler'>
VAE: <class 'diffusers.models.autoencoders.autoencoder_kl.AutoencoderKL'>

[Config Checks]
Bundle model id: SG161222/Realistic_Vision_V6.0_B1_noVAE
Bundle VAE id: stabilityai/sd-vae-ft-mse
VAE scaling factor: 0.18215
UNet in_channels: 4
UNet cross_attention_dim: 768
Scheduler num_train_timesteps: 1000
[After loading LOCAL bundle] allocated=10.21 GB | reserved=10.27 GB | max=10.21 GB

[OK] Built global_bundle and local_bundle
Global bundle name: Global_Realistic_Vision_V6.0_B1_noVAE
Local bundle name: Local_Realistic_Vision_V6.0_B1_noVAE

In [10]:
# Inject adapters + optimizers
mixed_global_bundle, mixed_local_bundle = build_mixed_lora_dora_training_setup(
    global_bundle=global_bundle,
    local_bundle=local_bundle,
    global_adapter_config={
        "adapter_type": "lora",
        "rank": 8,
        "alpha": 8,
        "dropout": 0.0,
        "target_suffixes": ["to_q", "to_k", "to_v", "to_out.0","ff.net.0.proj","ff.net.2"],
    },
    local_adapter_config={
        "adapter_type": "dora",
        "rank": 16,
        "alpha": 16,
        "dropout": 0.05,
        "target_suffixes": ["to_q", "to_k", "to_v", "to_out.0","ff.net.0.proj","ff.net.2"],
    },
    optimizer_config={
        "lr": 7e-5,
        "betas": (0.9, 0.999),
        "weight_decay": 1e-2,
    },
    freeze_before_injection=True,
    print_memory=True,
    print_reports=True,
    verbose=True,
)

[Before adapter injection] allocated=4.03 GB | reserved=4.08 GB | max=4.03 GB

[Applying LORA to existing UNet]
Bundle: Mixed_Global_LORA_Realistic_Vision_V6.0_B1_noVAE
Model id: SG161222/Realistic_Vision_V6.0_B1_noVAE
Rank: 8
Alpha: 8
Dropout: 0.0
Targets: ['to_q', 'to_k', 'to_v', 'to_out.0', 'ff.net.0.proj', 'ff.net.2']

Found Linear modules to LoRA-wrap: 160
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q                                 | 320->320
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k                                 | 320->320
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v                                 | 320->320
  down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0                             | 320->320
  down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_q                                 | 320->320
  down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_k                                 | 768->320
  down_block

In [75]:
# ScoreNet for local score loss
score_net = load_score_net_safely(
    checkpoint_path="models/score net/score_net_last_final.pt",
    device=str(device),
    dtype=torch.float32,
    base_channels=32,
    dropout=0.15,
    strict=True,
    freeze=True,
)



========== SAFE SCORENET LOAD ==========
checkpoint_path: models/score net/score_net_last_final.pt
target device: cuda
dtype: torch.float32
[Before loading ScoreNet] allocated=6.19 GB | reserved=10.27 GB | max=10.21 GB

[OK] ScoreNet loaded safely
freeze: True
eval_mode: True

[ScoreNet params]
total params:     439,561
trainable params: 0
frozen params:    439,561
[After loading ScoreNet] allocated=6.18 GB | reserved=6.69 GB | max=10.21 GB


In [76]:
# Local loss
local_loss = LDLALocalAgingLoss(
    local_bundle=mixed_local_bundle,
    score_net=score_net,

    lambda_full=1.0,
    lambda_zone=0.15,
    lambda_score=0.1,
    lambda_cycle=0.01,

    score_timestep_min=20,
    score_timestep_max=350,

    freeze_score_net=True,
    device=str(device),
)

[WARN] lambda_cycle > 0: cycle loss is expensive because it uses extra UNet passes. For the first local training run, consider lambda_cycle=0.0.


In [13]:
# Global auxiliary losses
global_aux_bundle = GlobalLossAuxBundle(
    device=str(device),
    dtype=torch.float32,
    use_age=True,
    age_model_id="nateraw/vit-age-classifier",
    age_image_size=224,
    use_identity=True,
    identity_pretrained="vggface2",
    identity_image_size=160,
    use_lpips=False,
)

preprocessor_config.json:   0%|          | 0.00/197 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[OK] Loaded HFAgeEstimator
model_id: nateraw/vit-age-classifier
num age classes: 9
age values: [1.0, 6.0, 14.5, 24.5, 34.5, 44.5, 54.5, 64.5, 70.0]


  0%|          | 0.00/107M [00:00<?, ?B/s]

[OK] Loaded FaceNetIdentityEncoder
pretrained: vggface2

========== GLOBAL LOSS AUX BUNDLE ==========
use_age: True
use_identity: True
use_lpips: False
device: cuda
dtype: torch.float32


In [14]:
# Global loss
global_loss = GlobalAgingLoss(
    global_bundle=mixed_global_bundle,
    global_loss_bundle=global_aux_bundle,
    lambda_diff=1.0,
    lambda_id=0.35,
    lambda_age=0.15,
    lambda_delta_age=0.15,
    lambda_perc=0.0,
    age_loss_scale=100.0,
    gamma_timestep=1.0,
    semantic_timestep_min=20,
    semantic_timestep_max=300,
    default_semantic_components=("age", "delta_age", "id"),
    device=str(device),
)


========== GLOBAL AGING LOSS ==========
lambda_diff: 1.0
lambda_id: 0.35
lambda_age: 0.15
lambda_delta_age: 0.15
lambda_perc: 0.0
age_loss_scale: 100.0
gamma_timestep: 1.0
timestep range: 0 999
semantic timestep range: 20 300
default_semantic_components: ('age', 'delta_age', 'id')
device: cuda
unet_dtype: torch.bfloat16


In [16]:
result = train_global_local_face_aging(
    mixed_local_bundle=mixed_local_bundle,
    mixed_global_bundle=mixed_global_bundle,

    # Main training loaders
    local_train_loader=local_train_loader,
    global_train_loader=global_train_loader,

    # Losses
    local_loss_fn=local_loss,
    global_loss_fn=global_loss,

    # Runtime
    device=device,
    amp_enabled=True,
    amp_dtype="bf16",

    # Run/checkpoints
    run_name="notebook_global_local_run",
    checkpoint_root="training_checkpoints/notebook_global_local_run",

    # Schedule
    num_epochs=6,
    local_num_epochs=6,
    global_num_epochs=6,
    train_order=("local", "global"),
    train_local=True,
    train_global=True,

    # Optimization
    local_grad_accum_steps=4,
    global_grad_accum_steps=4,
    local_grad_clip=1.0,
    global_grad_clip=1.0,

    # Local base loss sampling
    local_p_full=0.45,
    local_p_score=0.40,
    local_p_zone=0.15,
    local_enable_full=True,
    local_enable_score=True,
    local_enable_zone=True,
    local_p_neutral=0.05,
    local_p_double_full=0.20,

    # Optional local fused loss
    local_fused_train_loader=local_fused_train_loader,
    use_fused_loss=False,          # switch to True when ready
    fused_loss_epoch=15,
    fused_loss_every_n_steps=1,
    lambda_fuse_score=0.03,
    lambda_fuse_seam=0.01,
    fused_global_forward_fn=None,  # fallback x_global=x_orig; pass frozen global fn later

    # Global loss sampling
    global_p_diff=0.65,
    global_p_semantic=0.35,
    global_enable_diff=True,
    global_enable_semantic=True,
    global_semantic_components=("age", "delta_age", "id"),
    global_p_neutral=0.05,
    global_p_double_diff=0.15,
    min_target_age=18,
    max_target_age=90,

    # Schedulers
    build_schedulers_if_missing=True,
    local_warmup_ratio=0.05,
    global_warmup_ratio=0.05,
    local_min_lr=1e-6,
    global_min_lr=1e-6,
    min_warmup_steps=10,
    max_warmup_steps=None,

    # Checkpoints
    save_latest=True,
    save_best=True,
    save_inference_copy=True,
    local_monitor_key="loss/total",
    global_monitor_key="loss/total",

    # Memory
    enable_gradient_checkpointing_flag=True,
    offload_after_each_branch=True,
    print_memory=True,

    # Smoke controls
    local_max_batches=None,
    global_max_batches=None,

    # Fixed monitoring sample
    sampling_loader_global=sampling_loader_global,
    sampling_loader_local=sampling_loader_local,
    sample_every_epochs=1,
    sample_after_epoch_zero=False,
    sampling_output_dir="training_checkpoints/third_training_no_lips_v1/samples",

    # Sampling params
    sample_global_strength=0.30,
    sample_global_guidance_scale=4.5,
    sample_global_num_inference_steps=45,
    sample_global_negative_prompt=(
        "horror, zombie, corpse, skull, deformed face, distorted eyes, "
        "extreme wrinkles, diseased skin, low quality, artifacts"
    ),

    sample_local_strength=0.45,
    sample_local_guidance_scale=2.0,
    sample_local_num_inference_steps=45,
    sample_local_negative_prompt=(
        "blurry, smooth plastic skin, distorted skin, artifacts, low quality"
    ),
    sample_local_recycle_passes=2,
    sample_local_recycle_strength=0.07,
    
    # Fusion params
    sample_residual_alpha=0.40,
    sample_residual_sigma=7.5,
    sample_use_face_mask=True,
    sample_face_mask_blur_sigma=3.0,
    sample_local_insert_alpha=1.10,
    sample_local_mask_blur_sigma=4.0,
    sample_color_match=True,
    sample_color_match_strength=0.60,
    sample_seed=777,
    sample_save_grid=True,

    # Logging
    inner_print_every=300,
    inner_verbose=False,
    print_first_batch=False,
    verbose=True,
)

result


[Checkpoint managers created]
Root dir: training_checkpoints/notebook_global_local_run
Global dir: training_checkpoints/notebook_global_local_run/global
Local dir:  training_checkpoints/notebook_global_local_run/local

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Face Aging Global-Local Diffusion run: notebook_global_local_run
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Device    : cuda | AMP: True (bf16)
Schedule  : total_loop_epochs=6 | start_epoch=0 | local_epochs=6 | global_epochs=6
Local     : grad_accum=4 | p_full=0.45 | p_score=0.4 | p_zone=0.15 | p_double_full=0.2
Global    : grad_accum=4 | p_diff=0.65 | p_semantic=0.35 | p_double_diff=0.15
Sampling  : enabled=True | every=1 epochs | deterministic fusion only
Monitor   : train loss per branch (min) | checkpoints: LAST + BEST only
Checkpoints: training_checkpoints/notebook_global_local_run
───────

{'run_name': 'notebook_global_local_run',
 'history': {'local': [{'epoch': 0,
    'result': {'epoch': 0,
     'epoch_metrics': {'loss/loss': 0.09041303103295549,
      'loss/loss_full': 0.07888815799198146,
      'loss/loss_zone': 0.019678431946848488,
      'loss/loss_score': 0.06616723434186685,
      'loss/loss_cycle': 0.0,
      'loss/total': 0.09041303103295549,
      'loss_mode/full': 1.0,
      'optim/lr': 5.878900123341231e-05,
      'local/source_score_mean': 52.02672413793103,
      'local/source_score_std': 26.88077156132665,
      'local/source_score_ge_65': 0.38685344827586204,
      'local/source_score_ge_85': 0.25474137931034485,
      'local/source_score_ge_95': 0.12564655172413794,
      'local/target_score_mean': 74.69353448275862,
      'local/target_score_std': 17.703691674923075,
      'local/target_score_ge_65': 0.7625,
      'local/target_score_ge_85': 0.42995689655172414,
      'local/target_score_ge_95': 0.19913793103448277,
      'local/score_delta_mean': 22.6

In [77]:
# ============================================================
# LOCAL-ONLY TRAINING FROM EPOCH 0 TO EPOCH 4
# ============================================================

result = train_global_local_face_aging(
    mixed_local_bundle=mixed_local_bundle,
    mixed_global_bundle=mixed_global_bundle,

    # loaders
    local_train_loader=local_train_loader,
    global_train_loader=global_train_loader,

    # losses
    local_loss_fn=local_loss,
    global_loss_fn=None,

    # aux
    local_aux_objects=[score_net],
    global_aux_objects=[global_aux_bundle],

    # runtime
    device=device,
    amp_enabled=True,
    amp_dtype="bf16",

    # run/checkpoints
    run_name="local_only_dora_ffn_cycle_0_to_4",
    checkpoint_root="training_checkpoints/local_only_dora_ffn_cycle_0_to_4",

    # epochs 0,1,2,3,4
    num_epochs=5,
    local_num_epochs=5,
    global_num_epochs=0,
    start_epoch=0,

    train_order=("local",),
    train_local=True,
    train_global=False,

    # optimization
    local_grad_accum_steps=4,
    global_grad_accum_steps=4,
    local_grad_clip=1.0,
    global_grad_clip=1.0,

    # local loss sampling
    # Un poco menos score-extremo y un poco más zone para corregir regiones.
    local_p_full=0.40,
    local_p_score=0.35,
    local_p_zone=0.25,

    local_enable_full=True,
    local_enable_score=True,
    local_enable_zone=True,

    local_p_neutral=0.05,
    local_p_double_full=0.10,

    # fused OFF
    local_fused_train_loader=None,
    local_fused_loss_fn=None,
    use_fused_loss=False,
    fused_loss_epoch=999,
    fused_loss_every_n_steps=999,
    lambda_fuse_score=0.0,
    lambda_fuse_seam=0.0,
    fused_global_forward_fn=None,

    # global disabled
    global_p_diff=0.0,
    global_p_semantic=0.0,
    global_enable_diff=False,
    global_enable_semantic=False,
    global_semantic_components=("age", "delta_age", "id"),
    global_p_neutral=0.0,
    global_p_double_diff=0.0,
    min_target_age=18,
    max_target_age=85,

    # schedulers
    build_schedulers_if_missing=True,
    local_warmup_ratio=0.05,
    global_warmup_ratio=0.05,
    local_min_lr=1e-6,
    global_min_lr=1e-6,
    min_warmup_steps=10,
    max_warmup_steps=None,

    # checkpoints
    save_latest=True,
    save_best=True,
    save_inference_copy=True,
    local_monitor_key="loss/total",
    global_monitor_key="loss/total",

    # memory
    enable_gradient_checkpointing_flag=True,
    offload_after_each_branch=True,
    print_memory=True,

    # full local run
    local_max_batches=None,
    global_max_batches=0,

    # monitoring
    sampling_loader_global=sampling_loader_global,
    sampling_loader_local=sampling_loader_local,
    sample_every_epochs=1,
    sample_after_epoch_zero=False,
    sampling_output_dir="training_checkpoints/local_only_dora_ffn_cycle_0_to_4/samples",

    # sampling global
    sample_global_strength=0.34,
    sample_global_guidance_scale=4.6,
    sample_global_num_inference_steps=38,
    sample_global_negative_prompt=(
        "horror, zombie, corpse, skull, deformed face, distorted eyes, "
        "diseased skin, low quality, artifacts, plastic skin, waxy skin"
    ),

    # sampling local
    sample_local_strength=0.42,
    sample_local_guidance_scale=2.2,
    sample_local_num_inference_steps=45,
    sample_local_negative_prompt=(
        "blurry, distorted skin, artifacts, low quality, plastic skin, waxy skin"
    ),

    # fusion sampling
    sample_residual_alpha=0.60,
    sample_residual_sigma=6.0,
    sample_use_face_mask=True,
    sample_face_mask_blur_sigma=4.0,

    sample_local_insert_alpha=0.88,
    sample_local_mask_blur_sigma=7.0,

    sample_color_match=True,
    sample_color_match_strength=0.72,

    sample_seed=77,
    sample_save_grid=True,

    # logging
    inner_print_every=300,
    inner_verbose=False,
    print_first_batch=True,
    verbose=True,
)


[Checkpoint managers created]
Root dir: training_checkpoints/local_only_dora_ffn_cycle_0_to_4
Global dir: training_checkpoints/local_only_dora_ffn_cycle_0_to_4/global
Local dir:  training_checkpoints/local_only_dora_ffn_cycle_0_to_4/local

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Face Aging Global-Local Diffusion run: local_only_dora_ffn_cycle_0_to_4
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Device    : cuda | AMP: True (bf16)
Schedule  : total_loop_epochs=5 | start_epoch=0 | local_epochs=5 | global_epochs=0
Local     : grad_accum=4 | p_full=0.4 | p_score=0.35 | p_zone=0.25 | p_double_full=0.1
Global    : grad_accum=4 | p_diff=0.0 | p_semantic=0.0 | p_double_diff=0.0
Sampling  : enabled=True | every=1 epochs | deterministic fusion only
Monitor   : train loss per branch (min) | checkpoints: LAST + BEST only
Checkpoints: training_checkpoints/local_on

In [101]:
import importlib

import src.inference as inference_pkg
import src.training.training_sampling_helpers as sampling_helpers
import src.inference.inference_wrapper as inference_wrapper_mod

importlib.reload(inference_pkg)
importlib.reload(sampling_helpers)
importlib.reload(inference_wrapper_mod)

from src.inference.inference_wrapper import run_sampling_objects_inference

import importlib
import src.inference.fusion_bundle_maker as fusion_bundle_maker
import src.inference.inference_wrapper as inference_wrapper_mod

importlib.reload(fusion_bundle_maker)
importlib.reload(inference_wrapper_mod)

from src.inference.inference_wrapper import run_sampling_objects_inference
import importlib

import data.local_fused_dataset as local_fused_dataset
import data.create_data as create_data

import src.training.training_sampling_helpers as training_sampling_helpers
import src.inference.deterministic_fusion_ops as deterministic_fusion_ops
import src.inference.global_local_fusion as global_local_fusion
import src.inference.inference_wrapper as inference_wrapper

importlib.reload(local_fused_dataset)
importlib.reload(create_data)

importlib.reload(deterministic_fusion_ops)
importlib.reload(global_local_fusion)
importlib.reload(training_sampling_helpers)
importlib.reload(inference_wrapper)

from data.create_data import (
    build_local_fused_dataloaders,
    build_single_person_sampling_loader,
)

from src.inference.inference_wrapper import run_sampling_objects_inference
from src.training.training_sampling_helpers import run_deterministic_training_reconstruction_sample



sampling_objects = build_single_person_sampling_loader(
    image_stem="09501",
    target_age=62,
    skip=skip_zones,
    local_target_score={
        "default": 65,
        "frente":65,
        "surcos_nasogenianos": 65,
        "bajo_ojo_ojeras": 65,
        "patas_de_gallo": 65,
    },
    num_workers=0,
    pin_memory=False)

monitor_loader = sampling_objects["loader"]
sampling_loader_local = monitor_loader

[OK] JSON annotations already extracted: 56
[OK] Images already extracted: 56
[OK] Indexed unique images: 56
[OK] JSON files found: 56
[OK] Local annotated crop samples: 591


In [102]:

config_global_pass_push = {
    "generation": {
        "global_strength": 0.34,
        "global_guidance_scale": 4.6,
        "global_num_inference_steps": 40,
        "global_negative_prompt": (
            "horror, zombie, corpse, skull, deformed face, distorted eyes, "
            "diseased skin, low quality, artifacts, plastic skin, waxy skin"
        ),

        "local_strength": 0.46,
        "local_guidance_scale": 3.2,
        "local_num_inference_steps": 48,
        "local_negative_prompt": (
            "blurry, distorted skin, artifacts, low quality, plastic skin, waxy skin"
        ),

        "local_recycle_passes": 1,
        "seed": 77,
    },

    "fusion": {
        "residual_alpha": 0.62,
        "residual_sigma": 4.2,

        "residual_alpha_inside_local": 0.10,
        "residual_alpha_outside_local": 0.92,
        "local_union_blur_sigma": 6.0,

        "use_face_mask": True,
        "face_mask_blur_sigma": 4.0,

        "local_insert_alpha": 0.92,
        "local_mask_blur_sigma": 9.0,

        "color_match": True,
        "color_match_strength": 0.74,

        "fusion_prompt": "same person, realistic aged skin, natural wrinkles",
        "fusion_negative_prompt": (
            "changed identity, deformed face, artifacts, plastic skin, waxy skin"
        ),
    },

    "refiner": {
        "enabled": True,
        "strength": 0.03,
        "guidance_scale": 1.2,
        "num_inference_steps": 8,
        "prompt": (
            "realistic portrait photo of the same person, natural facial aging, "
            "preserve identity, realistic wrinkles"
        ),
        "negative_prompt": (
            "changed identity, distorted face, plastic skin, waxy skin, blurry, "
            "artifacts, over-smoothed skin"
        ),
    },

    "runtime": {
        "offload_after_each_stage": True,
        "return_pil": True,
        "save_grid": True,
        "verbose": True,
    },
}



result = run_sampling_objects_inference(
    sampling_objects=sampling_objects,
    mixed_global_bundle=mixed_global_bundle,
    mixed_local_bundle=mixed_local_bundle,

    checkpoint_paths={
        "local": "training_checkpoints/local_only_dora_ffn_cycle_0_to_4/local/epoch_003/adapter_inference.pt",
        "global": "training_checkpoints/notebook_global_local_run/global/epoch_005/adapter_inference.pt",
    },

    config=config_global_pass_push,
    output_dir="outputs/inference/09501_65anos",
)



[Inference checkpoint restored]
Path:           training_checkpoints/notebook_global_local_run/global/epoch_005/adapter_inference.pt
Branch:         global
Adapter loaded: 320

[Inference checkpoint restored]
Path:           training_checkpoints/local_only_dora_ffn_cycle_0_to_4/local/epoch_003/adapter_inference.pt
Branch:         local
Adapter loaded: 480

High-level sampling inference
Device        : cuda
Sample id     : 09501
Local zones   : 10
Refiner       : True
└─ [OFFLOAD] global inference modules moved to CPU
└─ [OFFLOAD] local inference modules moved to CPU


/venv/main/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/diffusers/pipelines/pipeline_utils.py:2267: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


[Refiner loaded] model=stabilityai/stable-diffusion-xl-refiner-1.0 | device=cuda | dtype=bfloat16 | strength=0.03 | guidance=1.2 | steps=8

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Global-local fusion
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Mode                 : model
Device               : cuda
Residual alpha       : 0.62
Residual alpha local : 0.1
Residual alpha outer : 0.92
Residual sigma       : 4.2
Local union sigma    : 6.0
Use face mask        : True
Local crops          : 10
Local insert α       : 0.92
Local mask sigma     : 9.0
Color match          : True
Color match strength : 0.74
Fusion model         : stabilityai/stable-diffusion-xl-refiner-1.0
Fusion strength      : 0.03
Fusion guidance      : 1.2
Fusion steps         : 8
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
[